**Pipeline (per cycle)**:
1. **Bbox Detection + Action Planning** — Qwen3-VL-4B sees the image, outputs object bbox + DSL action plan in one JSON
2. **3D Projection** — bbox center → ray-plane intersection → 3D world position
3. **Execution** — OSC_POSE controller executes action primitives in Robosuite
4. **Loop** — When action buffer empties, Qwen re-plans with updated 3D context

---
## 1. System Dependencies (EGL for headless MuJoCo)

In [1]:
import subprocess, os

missing = any(
    subprocess.run(f"dpkg -l {pkg}".split(), capture_output=True).returncode != 0
    for pkg in ["libegl1-mesa", "libegl1-mesa-dev", "libglib2.0-0"]
)
if missing:
    !sudo apt-get update -qq && sudo apt-get install -y -qq libegl1-mesa libegl1-mesa-dev libglib2.0-0
    print("System deps installed.")
else:
    print("System deps already present.")

os.environ["MUJOCO_GL"] = "egl"
os.environ["EGL_DEVICE_ID"] = "0"



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 10.)
debconf: falling back to frontend: Readline
Selecting previously unselected package libglx-dev:amd64.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../0-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../1-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../2-libegl-dev_1.4.0-1_amd64.deb ...
Unpacking libegl-dev:amd64 (1.4.0-1) ...
Selectin

---
## 2. Python Packages

In [2]:
# Install robosuite + MuJoCo + VLM dependencies
import subprocess, sys

def _pip_install(pkg_spec):
    """Run pip install and raise on failure."""
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkg_spec.split(),
        capture_output=True, text=True)
    if r.returncode != 0:
        print(f"pip install {pkg_spec} FAILED:")
        print(r.stderr[-500:])
        r.check_returncode()  # raise

_pip_install("mujoco>=3.3.0 robosuite>=1.5.2")
# numpy must be <2.1 for compatibility
_pip_install("numpy>=2.0,<2.1")
# Qwen3-VL needs transformers >= 4.57.0
_pip_install("transformers>=4.57.0 accelerate bitsandbytes sentencepiece einops timm qwen-vl-utils")
_pip_install("opencv-python-headless")

# Create dummy flash_attn (Qwen3-VL may check for it)
import os
_stub_dir = "/tmp/flash_attn"
os.makedirs(_stub_dir, exist_ok=True)
with open(f"{_stub_dir}/__init__.py", "w") as f:
    f.write('__version__ = "2.6.1"\n')
if "/tmp" not in sys.path:
    sys.path.insert(0, "/tmp")
import flash_attn
print(f"flash_attn stub OK (version {flash_attn.__version__})")

# Verify key packages
import mujoco
import robosuite as suite
import transformers
print(f"mujoco {mujoco.__version__}, robosuite {suite.__version__}, transformers {transformers.__version__}")
print("Python deps OK")

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (macros.py:59)


flash_attn stub OK (version 2.6.1)


[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


mujoco 3.9.0, robosuite 1.5.2, transformers 5.0.0
Python deps OK


---
## 3. Imports & GPU Check

In [3]:
import json, re, time, math, logging, os, sys, contextlib, warnings, copy, gc, random
from dataclasses import dataclass, field
from typing import Optional, List, Union, Any

import numpy as np
print(f"numpy {np.__version__}")

import torch

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore", module="robosuite")
for _name in ["robosuite", "robosuite.utils", "robosuite.controllers"]:
    logging.getLogger(_name).setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import robosuite as suite
print(f"robosuite {suite.__version__}")
from robosuite.controllers import load_composite_controller_config
import mujoco
print(f"mujoco {mujoco.__version__}")

SEED = None

def set_seed(seed: int):
    global SEED
    SEED = seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    mem = getattr(p, "total_memory", getattr(p, "total_mem", 0))
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {mem/1e9:.1f} GB")
else:
    print("No GPU — expect slow performance.")

print("Imports OK")

numpy 2.0.2
robosuite 1.5.2
mujoco 3.9.0
GPU: Tesla T4  VRAM: 15.6 GB
Imports OK


---
## 4. Config & Environment Setup


In [4]:
@dataclass
class Config:
    vlm_model_id: str = "Qwen/Qwen3-VL-4B-Instruct"
    use_4bit: bool = True
    env_name: str = "Lift"
    robot: str = "Panda"
    display_camera: str = "agentview"
    second_camera: str = "birdview"
    camera_size: int = 384
    feedback_interval_m: int = 4
    plan_length_n: int = 4
    lift_height: float = 0.15
    n_trials: int = 10
    max_steps: int = 150  # max env steps per trial
    seed: int = 42  # random seed for reproducibility

cfg = Config()
set_seed(cfg.seed)
print(f"VLM: {cfg.vlm_model_id}")
print(f"Feedback: every {cfg.feedback_interval_m} actions, plan {cfg.plan_length_n} actions, max {cfg.max_steps} steps")
print(f"Robot: {cfg.robot} in {cfg.env_name}")
print(f"Seed: {cfg.seed}")



VLM: Qwen/Qwen3-VL-4B-Instruct
Feedback: every 4 actions, plan 4 actions, max 150 steps
Robot: Panda in Lift
Seed: 42


---
## 5. DSL Specification: Robot Motion Description Language (RMDL)

### Formal Syntax (JSON-based)

Each action primitive is a JSON object. The VLM outputs a list of these primitives.

```
move_linear ::= { "type": "move_linear", "target": "object_above" | "ee" | [x, y, z],
                   "offset": [dx, dy, dz], "rotation": [dr, dp, dy](optional), "speed": float }
descend    ::= { "type": "descend", "speed": float }
grasp      ::= { "type": "grasp", "force": float }
lift       ::= { "type": "lift", "height": float, "speed": float }
release    ::= { "type": "release" }
wait       ::= { "type": "wait", "steps": int }
```

### Semantics

| Action | Behavior | Termination |
|--------|----------|-------------|
| move_linear (object_above) | Move EE to object_3d + offset, optional rotation | dist < 2.5cm or 60 steps |
| move_linear (absolute) | Move EE to [x,y,z], optional rotation | dist < 2.5cm or 60 steps |
| move_linear (ee) | Move EE relative to current position, optional rotation | dist < 2.5cm or 60 steps |
| descend | Press down while maintaining XY | plateau (dz<0.5mm for 12 st) or 150 steps |
| grasp | Close gripper + press down | fixed 40 steps |
| lift | Raise gripper above table | |dz| < 3cm or 70 steps |
| release | Open gripper | fixed 10 steps |
| wait | No-op | fixed N steps |

In [5]:
DSL_ACTION_TYPES = {"move_linear", "descend", "grasp", "lift", "release", "wait"}

def validate_action(action: dict) -> bool:
    if not isinstance(action, dict): return False
    atype = action.get("type")
    if atype not in DSL_ACTION_TYPES: return False
    if atype == "move_linear":
        target = action.get("target", "object_above")
        if isinstance(target, list):
            if len(target) != 3: return False
        elif target not in ("object_above", "ee"): return False
    if atype == "move_linear":
        offset = action.get("offset", [0, 0, 0])
        if not isinstance(offset, list) or len(offset) != 3 or not all(isinstance(v, (int, float)) for v in offset):
            return False
        rotation = action.get("rotation")
        if rotation is not None:
            if not isinstance(rotation, list) or len(rotation) != 3: return False
            if any(not isinstance(v, (int, float)) or v < -1 or v > 1 for v in rotation): return False
    if "speed" in action:
        s = action["speed"]
        if not isinstance(s, (int, float)) or s < 0.1 or s > 1.0: return False
    if atype == "grasp":
        f = action.get("force", 1.0)
        if not isinstance(f, (int, float)) or f < 0 or f > 1: return False
    if atype == "lift":
        h = action.get("height", 0.15)
        if not isinstance(h, (int, float)) or h < 0: return False
    if atype == "wait":
        st = action.get("steps", 10)
        if not isinstance(st, int) or st < 1: return False
    return True

def validate_plan(plan: dict) -> bool:
    if not isinstance(plan, dict): return False
    if "actions" not in plan or not isinstance(plan["actions"], list): return False
    if len(plan["actions"]) == 0: return False
    if not all(validate_action(a) for a in plan["actions"]): return False
    return True

def validate_bbox(bbox) -> bool:
    """Check bbox is exactly 4 normalized values (x1,y1,x2,y2), or None (omitted). x2>x1, y2>y1."""
    if bbox is None:
        return True
    if not isinstance(bbox, list): return False
    if len(bbox) != 4: return False
    if any(not isinstance(v, (int, float)) for v in bbox): return False
    if any(v < 0 or v > 1 for v in bbox): return False
    if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]: return False
    return True

def warn_bbox(plan: dict) -> None:
    """Print warning if bbox format is wrong."""
    bbox = plan.get("object_bbox")
    if bbox is not None and not validate_bbox(bbox):
        print(f"  [WARN] bad bbox (expected 4 values [x1,y1,x2,y2], got {len(bbox) if isinstance(bbox, list) else '?'}): {json.dumps(bbox)}")

_sample = {"reasoning": "test", "object_label": "cube", "object_bbox": [0.4, 0.4, 0.6, 0.6],
           "actions": [{"type": "move_linear", "target": "object_above", "offset": [0,0,0.15], "speed": 0.5}]}
assert validate_plan(_sample), "DSL validation failed"
print("DSL definition OK")



DSL definition OK


---
## 6. VLM Interface (Qwen3-VL)

In [6]:
class VLMPlanner:
    def __init__(self, model_id: str, use_4bit: bool = True):
        print(f"Loading Qwen3-VL: {model_id} ...")
        kwargs = {"torch_dtype": torch.float16, "device_map": "auto",
                  "trust_remote_code": True}
        if use_4bit and torch.cuda.is_available():
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
            kwargs["torch_dtype"] = torch.bfloat16
        self.model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_id, **kwargs).eval()
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self._call_count = 0
        print("  VLM loaded")

    def _call(self, images: dict, prompt: str,
              max_new_tokens: int = 1024) -> str:
        content = []
        for name in ["agentview", "birdview"]:
            if name in images:
                content.append({"type": "image", "image": images[name]})
        content.append({"type": "text", "text": prompt})
        conversation = [{"role": "user", "content": content}]
        inputs = self.processor.apply_chat_template(
            conversation, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        ).to(self.model.device)
        inputs.pop("token_type_ids", None)
        with torch.no_grad():
            if SEED is not None:
                torch.manual_seed(SEED + self._call_count)
                self._call_count += 1
            ids = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.3, top_p=0.9,
            )
        ids_trim = ids[0][inputs["input_ids"].shape[1]:]
        result = self.processor.decode(ids_trim, skip_special_tokens=True)
        print(f"  [VLM] raw:\n{result}\n", flush=True)
        if vlm_log_path:
            with open(vlm_log_path, 'a') as f:
                f.write(f"--- VLM Call #{self._call_count} ---\n{result}\n\n")
        return result

    # ─── Phase 1: Bbox detection ─────────────────────────────

    def _make_bbox_prompt(self, task):
        return f'''[ROLE] Robot vision system. Task: "{task}"

Two camera images are provided — you MUST analyze BOTH:

1. agentview (front view — VERTICALLY FLIPPED):
   UP in image = high Z (arm). DOWN = table surface.
   LEFT = robot LEFT (+Y). RIGHT = robot RIGHT (-Y).
   → Use this view for bbox output.

2. birdview (top-down):
   X = horizontal, Y = vertical on table.
   → Cross-check with agentview to confirm object location.

Output ONLY this JSON:
{{"reasoning":"precise observation","object_bbox":[x1,y1,x2,y2]}}

RULES:
- reasoning: brief, precise. No long explanations.
- object_bbox: 4 values [x1,y1,x2,y2] normalized 0-1 from agentview. x2>x1, y2>y1.
- Set to null only if fully occluded.
- JSON ONLY. No text outside JSON.'''

    def query_bbox(self, image, task, extra_images=None, max_retries=2):
        images = {"agentview": Image.fromarray(image)}
        if extra_images:
            for k, v in extra_images.items():
                images[k] = Image.fromarray(v)
        prompt = self._make_bbox_prompt(task)
        for attempt in range(max_retries + 1):
            raw = self._call(images, prompt, max_new_tokens=512)
            plan = self._extract_json(raw)
            if plan is not None:
                bbox = plan.get("object_bbox")
                if bbox is not None and validate_bbox(bbox):
                    rsn = plan.get("reasoning", "") or ""
                    print(f"  [BBOX] {rsn[:120]}", flush=True)
                    print(f"  [BBOX] {bbox}", flush=True)
                    return bbox, rsn
            print(f"  [BBOX] attempt {attempt+1} failed", flush=True)
            if attempt < max_retries:
                prompt += "\n\nInvalid. Return JSON with reasoning and object_bbox."
        return None

    # ─── Phase 2: Action planning ────────────────────────────

    def _make_initial_prompt(self, task, object_3d):
        if object_3d is None:
            obj_str = "(0.000, 0.000, 0.800)"
        else:
            obj_str = f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})"
        return (
            f'[ROLE] Robot arm controller. Task: "{task}"\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            '\n'
            f'[TARGET POSITION] Object at {obj_str}\n'
            + _SHARED_BODY
            + _STRATEGY + '\n'
            + '[RULES]\n'
            '1. Plan 3-8 actions. Use "object_above"+offset to approach the target (offset [0,0,0.1] to hover 10 cm above).\n'
            + _SHARED_RULES
        )

    def _make_closedloop_prompt(self, task, steps_done, completed_actions, ee_pos, n_actions,
                                object_3d, gripper_state="unknown"):
        if object_3d is None:
            obj_str = "(0.000, 0.000, 0.800)"
        else:
            obj_str = f"({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})"
        completed_str = "; ".join(completed_actions[-5:]) if completed_actions else "none yet"
        return (
            f'[STATUS] Step {steps_done}/{cfg.max_steps}. Task "{task}" is still NOT complete.\n'
            f'{len(completed_actions)} actions have been executed, but the cube has NOT been lifted yet.\n'
            f'You MUST output exactly {n_actions} actions to finally succeed.\n'
            '\n'
            'Robot: Panda 7-DOF + parallel gripper. Controller: OSC_POSE (EE position control).\n'
            + _SHARED_BODY
            + '[STATE]\n'
            f'  EE position: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})\n'
            f'  Target 3D: {obj_str}\n'
            f'  Gripper: {gripper_state}\n'
            f'  Recent history: {completed_str}\n'
            '\n'
            + '[RULES — STRICT]\n'
            f'1. Output EXACTLY {n_actions} actions. Empty list or wrong count → REJECTED.\n'
            '2. Use "object_above"+offset to approach the target.\n'
            + _SHARED_RULES
        )

    def plan_initial(self, image, task, object_3d, max_retries=2, extra_images=None):
        images = {"agentview": Image.fromarray(image)}
        if extra_images:
            for k, v in extra_images.items():
                images[k] = Image.fromarray(v)
        prompt = self._make_initial_prompt(task, object_3d)
        for attempt in range(max_retries + 1):
            raw = self._call(images, prompt, max_new_tokens=768)
            plan = self._extract_json(raw)
            if plan is not None and validate_plan(plan):
                rsn = plan.get("reasoning", "")
                if rsn:
                    print(f"  [PLAN] reasoning: {rsn[:200]}", flush=True)
                print(f"  [PLAN] {len(plan['actions'])} actions", flush=True)
                return plan
            print(f"  [PLAN] attempt {attempt+1} failed", flush=True)
            if attempt < max_retries:
                prompt += "\n\nInvalid. Return valid JSON with reasoning and actions."
        return None

    def plan_closedloop(self, image, task, steps_done, completed_actions, ee_pos, n_actions,
                        object_3d, gripper_state="unknown", extra_images=None):
        images = {"agentview": Image.fromarray(image)}
        if extra_images:
            for k, v in extra_images.items():
                images[k] = Image.fromarray(v)
        prompt = self._make_closedloop_prompt(
            task, steps_done, completed_actions, ee_pos, n_actions,
            object_3d, gripper_state=gripper_state)
        attempt = 0
        while True:
            raw = self._call(images, prompt, max_new_tokens=768)
            plan = self._extract_json(raw)
            if plan is not None and validate_plan(plan):
                if len(plan["actions"]) == n_actions:
                    rsn = plan.get("reasoning", "")
                    if rsn:
                        print(f"    [PLAN] reasoning: {rsn[:200]}", flush=True)
                    print(f"    [PLAN] {len(plan['actions'])} actions", flush=True)
                    return plan
                err = f"expected {n_actions} actions, got {len(plan['actions'])}"
            elif plan is None:
                err = "json_parse_failed"
            elif not isinstance(plan.get("actions"), list):
                err = "no_actions"
            else:
                err = next((f"invalid_action_{i}: {json.dumps(a)}" for i, a in enumerate(plan["actions"]) if not validate_action(a)), "unknown")
            print(f"    [PLAN] retry {attempt+1}: {err}", flush=True)
            attempt += 1

    @staticmethod
    def _extract_json(text: str) -> Optional[dict]:
        text = text.strip()
        if text.startswith("{"):
            try:
                result = json.loads(text)
            except json.JSONDecodeError:
                result = None
            if result is not None:
                # Duplicate "actions" key workaround: VLM sometimes outputs
                # "actions":[] then appends "actions":[{real}] during generation.
                # json.loads keeps the LAST occurrence, but empty-only is broken.
                # Scan raw text for ALL actions arrays and pick first non-empty one.
                if isinstance(result.get("actions"), list) and len(result["actions"]) == 0:
                    _pat = re.compile(r'"actions"\s*:\s*(\[)', re.DOTALL)
                    for _m in _pat.finditer(text):
                        _start = _m.start(1)
                        _depth, _i = 1, _start + 1
                        while _depth > 0 and _i < len(text):
                            if text[_i] == '[':
                                _depth += 1
                            elif text[_i] == ']':
                                _depth -= 1
                            _i += 1
                        try:
                            _alt = json.loads(text[_start:_i])
                            if isinstance(_alt, list) and len(_alt) > 0:
                                result["actions"] = _alt
                                break
                        except (json.JSONDecodeError, TypeError):
                            continue
                return result
        if text.startswith("["):
            try:
                arr = json.loads(text)
                if isinstance(arr, list) and len(arr) > 0:
                    if all(isinstance(item, dict) and "type" in item for item in arr):
                        return {"actions": arr}
            except (json.JSONDecodeError, TypeError):
                pass
        m = re.search(r"\{(?:[^{}]|\{[^{}]*\})*\}", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
        start = text.find("{")
        if start >= 0:
            depth = 0
            for i in range(start, len(text)):
                if text[i] == "{": depth += 1
                elif text[i] == "}": depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i+1])
                    except json.JSONDecodeError:
                        pass
                    break
        start = text.find("[")
        if start >= 0:
            depth = 0
            for i in range(start, len(text)):
                if text[i] == "[": depth += 1
                elif text[i] == "]": depth -= 1
                if depth == 0:
                    try:
                        arr = json.loads(text[start:i+1])
                        if isinstance(arr, list) and len(arr) > 0:
                            if all(isinstance(item, dict) and "type" in item for item in arr):
                                return {"actions": arr}
                    except (json.JSONDecodeError, TypeError):
                        pass
                    break
        return None

    def unload(self):
        del self.model
        torch.cuda.empty_cache()
        print("  VLM unloaded")

In [7]:
# ═══════════════════════════════════════════════════════════
# Shared prompt constants (identical in initial + closed-loop prompts)
# ═══════════════════════════════════════════════════════════

_CAMERA_DESC = (
    '[CAMERA VIEWS — TWO images provided, you MUST analyze BOTH]\n'
    '1. agentview (front, VERTICALLY FLIPPED):\n'
    '   UP in image = high Z (robot arm, above the table). DOWN in image = table surface.\n'
    '   LEFT in image = robot\'s LEFT (+Y). RIGHT in image = robot\'s RIGHT (-Y).\n'
    '   The arm appears at the top of the image reaching down toward the table at the bottom.\n'
    '   → Use this view for bbox output and to judge depth/height.\n'
    '\n'
    '2. birdview (top-down):\n'
    '   Camera looks straight down at the table from above.\n'
    '   X axis = horizontal in image, Y axis = vertical in image.\n'
    '   → Use this view for spatial layout, XY positions, and object-arm distance.\n'
    '\n'
    'CRITICAL: Both images together give you complete 3D understanding. '
    'agentview alone cannot show XY position accurately; '
    'birdview alone cannot show height. You NEED both.\n'
)

_COORD_DESC = (
    '[COORDINATE SYSTEM]\n'
    'X = forward (+ = away from robot), Y = robot-left (+ = left), Z = up (table surface = 0.80)\n'
)

_ACTION_REFERENCE = (
    '[ACTIONS — complete reference]\n'
    'All actions are JSON objects with "type" and type-specific parameters.\n'
    '\n'
    '1. move_linear — Move end-effector to a target position\n'
    '   {"type":"move_linear", "target":"object_above"|"ee"|[x,y,z], "offset":[dx,dy,dz], "rotation":[dr,dp,dy], "speed":float, "max_steps":int}\n'
    '   target: "object_above" = detected 3D pos; "ee" = relative EE; [x,y,z] = absolute meters\n'
    '   offset: [dx,dy,dz] meters added to target (default [0,0,0.15])\n'
    '   rotation: [dr,dp,dy] each in [-1,1] (optional)\n'
    '   speed: 0.1-1.0 (default 0.5); max_steps: 8-60 (default 60)\n'
    '   → Stops early when within 2.5 cm of target\n'
    '\n'
    '2. descend — Press gripper downward until physically blocked (plateau)\n'
    '   {"type":"descend", "speed":float, "max_steps":int}\n'
    '   speed: 0.1-0.5 (default 0.3); max_steps: 20-150 (default 150)\n'
    '   → Maintains XY over target. Auto-stops when motion <0.5mm for 12 steps.\n'
    '\n'
    '3. grasp — Close gripper to grab object\n'
    '   {"type":"grasp", "force":float, "max_steps":int}\n'
    '   force: 0.0-1.0 (default 1.0); max_steps: 8-40 (default 40)\n'
    '\n'
    '4. lift — Raise EE to lift grasped object above table\n'
    '   {"type":"lift", "height":float, "speed":float, "max_steps":int}\n'
    '   height: meters above table (default 0.15); speed: 0.1-1.0 (default 0.5); max_steps: 5-70 (default 70)\n'
    '   → Lifts to table_z + height. Stops within 3 cm of target.\n'
    '\n'
    '5. release — Open gripper (always 10 steps)\n'
    '   {"type":"release"}\n'
    '\n'
    '6. wait — Do nothing for N steps\n'
    '   {"type":"wait", "steps":int}  steps: integer >= 1\n'
)

_OUTPUT_FORMAT = (
    '[OUTPUT FORMAT]\n'
    '{"reasoning":"1-2 sentence strategy","actions":[{action1},{action2},...]}\n'
)

_SHARED_BODY = (
    '\n' + _CAMERA_DESC + '\n'
    + _COORD_DESC + '\n'
    + _ACTION_REFERENCE + '\n'
    + _OUTPUT_FORMAT + '\n'
)

_SHARED_RULES = (
    '- CRITICAL: Analyze BOTH agentview (for depth/height) AND birdview (for XY layout) together.\n'
    '- CRITICAL: The task is NOT yet complete — you MUST continue executing actions.\n'
    '- CRITICAL: NEVER output "actions":[]. There is always a valid action (descend, grasp, lift, move_linear). Empty array → REJECTED.\n'
    '- max_steps by distance: short=8-20, medium=20-40, long=40-60.\n'
    '- Speed: 0.2-0.4 for precision (descend/grasp), 0.5-0.8 for gross movement (move_linear/lift).\n'
    '- Vary action types — avoid consecutive repeats of the same type.\n'
    '- JSON ONLY. No text outside the JSON. Every field must be valid JSON.\n'
)

_STRATEGY = (
    '[STRATEGY for pick-up]\n'
    'Standard approach: move_linear(hover above) → descend(press down) → grasp(close) → lift(raise).\n'
    'Add extra alignment moves if the arm is far from the target.\n'
)

In [8]:
vlm_log_path = None

_vlm_planner = None

def get_planner():
    global _vlm_planner
    if _vlm_planner is None:
        gc.collect()
        torch.cuda.empty_cache()
        _vlm_planner = VLMPlanner(cfg.vlm_model_id, cfg.use_4bit)
    return _vlm_planner

print("VLMPlanner class defined")

VLMPlanner class defined


---
## 7. Geometry Utilities

In [9]:
# Birdview camera offset: lowered by this many meters (positive = lower)
BIRDVIEW_Z_OFFSET = 1.2


def adjust_birdview_fov(env):
    """Narrow birdview FOV to simulate a lower camera position.
    Must be called AFTER env.reset() since reset reinitializes cam_fovy."""
    try:
        bv_id = env.sim.model.camera_name2id("birdview")
        if bv_id < 0:
            return
        cam_z = float(env.sim.model.cam_pos[bv_id, 2])
        tz = float(env.model.mujoco_arena.table_offset[2])
        old_dist = cam_z - tz
        new_dist = old_dist - BIRDVIEW_Z_OFFSET
        zf = old_dist / new_dist
        old_fov = float(env.sim.model.cam_fovy[bv_id])
        new_fov = 2.0 * math.degrees(math.atan(math.tan(math.radians(old_fov) / 2.0) / zf))
        env.sim.model.cam_fovy[bv_id] = new_fov
        print(f"  Birdview: Z={cam_z:.3f}, table Z={tz:.3f}, offset={BIRDVIEW_Z_OFFSET}m", flush=True)
        print(f"  Birdview FOV: {old_fov:.1f}° -> {new_fov:.1f}° (zoom={zf:.2f}x)", flush=True)
    except Exception as e:
        print(f"  [WARN] birdview FOV adjust failed: {e}", flush=True)


def make_env(cfg: Config, seed: int = None):
    cameras = list(dict.fromkeys([cfg.display_camera, cfg.second_camera]))
    cc = load_composite_controller_config(controller="BASIC")
    with open(os.devnull, "w") as _dn:
        with contextlib.redirect_stdout(_dn), contextlib.redirect_stderr(_dn):
            env = suite.make(
                cfg.env_name, robots=cfg.robot, controller_configs=cc,
                has_renderer=False, has_offscreen_renderer=True, renderer="mujoco",
                camera_names=cameras,
                camera_heights=cfg.camera_size, camera_widths=cfg.camera_size,
                camera_depths=False,
                use_camera_obs=True, reward_shaping=True, control_freq=20,
                seed=seed if seed is not None else cfg.seed,
            )
    return env


def render_rgb(env, camera="agentview") -> np.ndarray:
    obs = env._get_observations(force_update=True)
    # Flip vertically so VLM sees arm-at-top, cube-at-bottom (matches physical up/down)
    return np.flipud(obs[f"{camera}_image"][..., :3].astype(np.uint8))


def unflip_bbox(bbox):
    if bbox is None or not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        return None
    return [bbox[0], 1.0 - bbox[3], bbox[2], 1.0 - bbox[1]]


def get_camera_params(env, camera="agentview", size=384):
    cam_id = env.sim.model.camera_name2id(camera)
    fov = float(env.sim.model.cam_fovy[cam_id])
    f = size / (2.0 * math.tan(math.radians(fov) / 2.0))
    K = np.array([[f, 0, size / 2], [0, f, size / 2], [0, 0, 1]])
    cam_pos = env.sim.model.cam_pos[cam_id].copy()
    cam_R_raw = env.sim.model.cam_mat0[cam_id].reshape(3, 3).copy()
    table_z = float(env.model.mujoco_arena.table_offset[2])
    return K, cam_pos, cam_R_raw, table_z


def bbox_to_ray(bbox, K, cam_pos, cam_R):
    cu = (bbox[0] + bbox[2]) / 2.0
    cv = bbox[3]
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    d = cam_R @ np.array([(cu - cx) / fx, -(cv - cy) / fy, -1.0])
    d = d / np.linalg.norm(d)
    return cam_pos.copy(), d


def bbox_to_3d(bbox_norm, img_size, env, camera="agentview"):
    h, w = img_size, img_size
    if bbox_norm is None or not isinstance(bbox_norm, (list, tuple)) or len(bbox_norm) != 4:
        return None
    bbox_abs = [bbox_norm[0] * w, bbox_norm[1] * h,
                bbox_norm[2] * w, bbox_norm[3] * h]
    K, cam_pos, cam_R, tz = get_camera_params(env, camera=camera)
    origin, direction = bbox_to_ray(bbox_abs, K, cam_pos, cam_R)
    t = (tz - origin[2]) / direction[2]
    if t <= 0: return None
    p3d = origin + t * direction
    p3d[2] = tz
    return p3d


# Quick test
cc = load_composite_controller_config(controller="BASIC")
with open(os.devnull, "w") as _dn:
    with contextlib.redirect_stdout(_dn), contextlib.redirect_stderr(_dn):
        test_env = suite.make(
            cfg.env_name, robots=cfg.robot, controller_configs=cc,
            has_renderer=False, has_offscreen_renderer=True, renderer="mujoco",
            camera_names=["agentview", "birdview"],
            camera_heights=cfg.camera_size, camera_widths=cfg.camera_size,
            camera_depths=False,
            use_camera_obs=True, reward_shaping=True, control_freq=20,
            seed=cfg.seed,
        )
bv_idx = test_env.sim.model.camera_name2id("birdview")
test_env.reset()

# Render before
before = render_rgb(test_env, camera="birdview")

# Apply FOV adjustment (same as adjust_birdview_fov)
cam_z = float(test_env.sim.model.cam_pos[bv_idx, 2])
tz = float(test_env.model.mujoco_arena.table_offset[2])
old_fov = float(test_env.sim.model.cam_fovy[bv_idx])
zf = (cam_z - tz) / (cam_z - tz - BIRDVIEW_Z_OFFSET)
new_fov = 2.0 * math.degrees(math.atan(math.tan(math.radians(old_fov) / 2.0) / zf))
test_env.sim.model.cam_fovy[bv_idx] = new_fov
print(f"  FOV: {old_fov:.1f}° -> {new_fov:.1f}° (zoom={zf:.2f}x)", flush=True)

# Render after
after = render_rgb(test_env, camera="birdview")

diff = np.abs(before.astype(np.int16) - after.astype(np.int16))
diff_pixels = np.sum(diff > 5)
total = 384 * 384 * 3
print(f"  Pixels changed (|diff|>5): {diff_pixels}/{total} ({100*diff_pixels/total:.1f}%)", flush=True)
print(f"  {'OK: FOV works' if diff_pixels > 0 else 'WARNING: no effect!'}")

ccam = []
for i in range(test_env.sim.model.ncam):
    name = test_env.sim.model.camera_names[i]
    if isinstance(name, bytes): name = name.decode()
    ccam.append(name)
cube_pos = test_env.sim.data.body_xpos[test_env.cube_body_id].copy()
test_env.close()

print(f"Available cameras: {ccam}")
print(f"Cube pos: ({cube_pos[0]:.3f}, {cube_pos[1]:.3f}, {cube_pos[2]:.3f})")
print("Geometry utils OK")

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but 

  FOV: 45.0° -> 21.3° (zoom=2.20x)
  Pixels changed (|diff|>5): 363095/442368 (82.1%)
  OK: FOV works
Available cameras: ['frontview', 'birdview', 'agentview', 'sideview', 'robot0_robotview', 'robot0_eye_in_hand']
Cube pos: (0.024, 0.017, 0.830)
Geometry utils OK


---
## 8. VLM Controller (Closed-Loop)

In [10]:
from dataclasses import dataclass, field
@dataclass
class TrialResult:
    success: bool
    steps: int
    error: Optional[str] = None
    plan: Optional[dict] = None

In [11]:
class VideoCapture:
    def __init__(self, fps=4):
        self.frames = []
        self.fps = fps
        try:
            self._font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
        except Exception:
            self._font_path = None
    def snap(self, img, phase="", plan=None, t3d=None, msg="", step=0, pause=False, img2=None, bbox=None):
        import cv2
        from PIL import Image, ImageDraw, ImageFont
        img = np.ascontiguousarray(img)
        if bbox is not None and len(bbox) == 4:
            hh, ww = img.shape[:2]
            x1 = int(bbox[0] * ww)
            y1 = int(bbox[1] * hh)
            x2 = int(bbox[2] * ww)
            y2 = int(bbox[3] * hh)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        if img2 is not None:
            bar_h = 20
            def _add_label(img_np, label):
                hh, ww = img_np.shape[:2]
                frame = np.zeros((hh + bar_h, ww, 3), dtype=np.uint8)
                frame[:hh] = img_np
                frame[hh:][:] = (40, 40, 50)
                pil_f = Image.fromarray(frame)
                try:
                    font_b = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
                except Exception:
                    font_b = ImageFont.load_default()
                ImageDraw.Draw(pil_f).text((6, hh + 2), label, font=font_b, fill=(200, 230, 255))
                return np.array(pil_f)
            left = _add_label(img, "agentview")
            right = _add_label(img2, "birdview")
            disp = np.concatenate([left, right], axis=1)
        else:
            disp = img.copy()
        h, w = disp.shape[:2]
        info_h = 260
        composite = np.zeros((h + info_h, w, 3), dtype=np.uint8)
        composite[:h] = disp
        composite[h:][:] = (25, 25, 35)
        pil_img = Image.fromarray(composite)
        draw = ImageDraw.Draw(pil_img)
        try:
            font_m = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
            font_s = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
        except Exception:
            font_m = font_s = ImageFont.load_default()
        def _wrap(text, font, max_w):
            words = text.split()
            lines_o = []; cur = ""
            for wd in words:
                test = cur + " " + wd if cur else wd
                if draw.textbbox((0, 0), test, font=font)[2] <= max_w: cur = test
                else:
                    if cur: lines_o.append(cur)
                    cur = wd
            if cur: lines_o.append(cur)
            return lines_o
        margin = 10
        max_text_w = w - 2 * margin
        y = h + 8
        draw.text((margin, y), f"Step {step}  {phase}", font=font_m, fill=(255, 200, 100))
        y += 22
        if plan:
            for rl in _wrap(plan.get("reasoning", ""), font_s, max_text_w):
                draw.text((margin, y), rl, font=font_s, fill=(180, 220, 255)); y += 16
            for aidx, act in enumerate(plan.get("actions", [])):
                for al in _wrap(f"  {aidx+1}. {json.dumps(act, ensure_ascii=False)}", font_s, max_text_w):
                    draw.text((margin, y), al, font=font_s, fill=(150, 200, 150)); y += 16
                    if y > h + info_h - 16: break
                if y > h + info_h - 16:
                    draw.text((margin, y), "  ...", font=font_s, fill=(100, 100, 100)); break
        if t3d is not None:
            draw.text((margin, y), f"3D: ({t3d[0]:.3f}, {t3d[1]:.3f}, {t3d[2]:.3f})", font=font_s, fill=(150, 200, 255))
            y += 16
        if msg:
            for ml in _wrap(msg, font_s, max_text_w):
                draw.text((margin, y), ml, font=font_s, fill=(200, 200, 200)); y += 16
        self.frames.append(np.array(pil_img))
        if pause:
            for _ in range(3): self.frames.append(np.array(pil_img))

    def save(self, path="output.mp4"):
        n_frames = len(self.frames)
        print(f"  [SAVE] {n_frames} frames, {self.fps} fps", flush=True)
        if not self.frames: return path
        h, w = self.frames[0].shape[:2]
        import cv2
        for codec, ext in [('avc1', '.mp4'), ('H264', '.mp4'), ('X264', '.mp4'),
                           ('mp4v', '.mp4'), ('XVID', '.avi')]:
            fpath = path.rsplit('.', 1)[0] + ext if ext != '.mp4' else path
            try:
                fourcc = cv2.VideoWriter_fourcc(*codec)
                out = cv2.VideoWriter(fpath, fourcc, self.fps, (w, h))
                if not out.isOpened():
                    out.release(); continue
                for f in self.frames:
                    out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
                out.release()
                size = os.path.getsize(fpath)
                if size < max(1024, n_frames * w * h // 10000):
                    print(f"  [SAVE] {codec} too small, trying next", flush=True)
                    if fpath != path:
                        try: os.remove(fpath)
                        except: pass
                    continue
                print(f"  [SAVE] Written with {codec}: {size/1024:.0f} KB", flush=True)
                return fpath
            except Exception:
                continue
        print("  [SAVE] All codecs failed!", flush=True)
        return path

In [12]:
class VLMController:
    def __init__(self, env, planner: VLMPlanner, on_frame=None):
        self.env = env
        self.planner = planner
        eef = env.robots[0].eef_site_id
        self._ee_site = eef[list(eef.keys())[0]] if isinstance(eef, dict) else eef
        self._tz = 0.800
        self.on_frame = on_frame
        self._step = 0
        self._action_count = 0
        self._action_type = None
        self._object_3d = None
        self._bbox = None
    def _ee_pos(self):
        return self.env.sim.data.site_xpos[self._ee_site].copy()
    def _osc_action(self, dxyz, grip, rotation=None):
        if rotation is None:
            rotation = [0, 0, 0]
        return np.concatenate([np.clip(dxyz / 0.05, -1, 1), np.clip(np.array(rotation, dtype=float), -1, 1), [grip]])
    def _execute_action(self, action: dict, object_3d: np.ndarray) -> str:
        atype = action["type"]
        speed = action.get("speed", 0.5)
        if atype == "move_linear":
            target = action.get("target", "object_above")
            if isinstance(target, str) and target == "object_above":
                offset = action.get("offset", [0, 0, 0.15])
                target_pos = object_3d + np.array(offset)
            elif isinstance(target, str) and target == "ee":
                offset = action.get("offset", [0, 0, 0])
                target_pos = self._ee_pos() + np.array(offset)
            elif isinstance(target, list):
                target_pos = np.array(target, dtype=float)
            else:
                target_pos = object_3d + np.array([0, 0, 0.15])
            rotation = action.get("rotation")
            max_st = max(action.get("max_steps", 60), 20)
            for _ in range(max_st):
                self._step += 1
                ee = self._ee_pos()
                if np.linalg.norm(ee - target_pos) < 0.025:
                    self.env.step(np.zeros(7))
                    break
                d = target_pos - ee
                act = self._osc_action(d * speed, -1.0, rotation)
                self.env.step(act)
                if self.env.sim.data.body_xpos[self.env.cube_body_id][2] > self._tz + 0.05:
                    if self.on_frame:
                        self.on_frame(render_rgb(self.env), "SUCCESS",
                                      None, None, f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
                    return "task_done"
                if self.on_frame:
                    self.on_frame(render_rgb(self.env), f"Move #{self._action_count}: {atype}",
                                  None, target_pos, f"dist={np.linalg.norm(ee-target_pos):.3f}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
            return "reached" if np.linalg.norm(self._ee_pos() - target_pos) < 0.025 else "max_steps"
        elif atype == "descend":
            speed = action.get("speed", 0.3)
            max_st = max(action.get("max_steps", 150), 20)
            target_xy = object_3d[:2].copy()
            stagnant = 0
            prev_z = self._ee_pos()[2]
            for i in range(min(max_st, 150)):
                self._step += 1
                ee = self._ee_pos()
                dz = abs(ee[2] - prev_z)
                if dz < 0.0005:
                    stagnant += 1
                    if stagnant > 12:
                        self.env.step(np.zeros(7))
                        break
                else:
                    stagnant = 0
                prev_z = ee[2]
                d = np.array([target_xy[0] - ee[0], target_xy[1] - ee[1], -0.3])
                act = self._osc_action(d * speed, -1.0, action.get("rotation"))
                self.env.step(act)
                if self.env.sim.data.body_xpos[self.env.cube_body_id][2] > self._tz + 0.05:
                    if self.on_frame:
                        self.on_frame(render_rgb(self.env), "SUCCESS",
                                      None, None, f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
                    return "task_done"
                if self.on_frame:
                    self.on_frame(render_rgb(self.env), f"Move #{self._action_count}: {atype}",
                                  None, None, f"z={ee[2]:.3f} stagnant={stagnant}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
            return "plateau"
        elif atype == "grasp":
            force = action.get("force", 1.0)
            max_st = max(action.get("max_steps", 40), 8)
            for i in range(max_st):
                self._step += 1
                self.env.step(np.array([0, 0, 0, 0, 0, 0, force]))
                if self.env.sim.data.body_xpos[self.env.cube_body_id][2] > self._tz + 0.05:
                    return "task_done"
                if self.on_frame:
                    self.on_frame(render_rgb(self.env), f"Move #{self._action_count}: {atype}",
                                  None, None, f"force={force}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
            return "done"
        elif atype == "lift":
            height = action.get("height", 0.15)
            speed = action.get("speed", 0.5)
            target_z = self._tz + height
            ee_start = self._ee_pos()
            max_st = max(action.get("max_steps", 70), 5)
            for _ in range(max_st):
                self._step += 1
                ee = self._ee_pos()
                if abs(ee[2] - target_z) < 0.03:
                    self.env.step(np.zeros(7))
                    break
                d = np.array([ee_start[0] - ee[0], ee_start[1] - ee[1], target_z - ee[2]])
                act = self._osc_action(d * speed, 1.0)
                self.env.step(act)
                if self.env.sim.data.body_xpos[self.env.cube_body_id][2] > self._tz + 0.05:
                    if self.on_frame:
                        self.on_frame(render_rgb(self.env), "SUCCESS",
                                      None, None, f"cube lifted to {self.env.sim.data.body_xpos[self.env.cube_body_id][2]:.3f}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
                    return "task_done"
                if self.on_frame:
                    self.on_frame(render_rgb(self.env), f"Move #{self._action_count}: {atype}",
                                  None, None, f"z={ee[2]:.3f}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
            return "at_height" if abs(self._ee_pos()[2] - target_z) < 0.03 else "max_steps"
        elif atype == "release":
            for _ in range(10):
                self._step += 1
                self.env.step(np.array([0, 0, 0, 0, 0, 0, -1.0]))
            return "done"
        elif atype == "wait":
            for _ in range(action.get("steps", 10)):
                self._step += 1
                self.env.step(np.zeros(7))
            return "done"
        return "unknown"

    def _detect_and_project(self, task: str) -> tuple:
        """Run Phase 1: bbox detection → 3D projection. Returns (object_3d, bbox) or (None, None)."""
        img = render_rgb(self.env)
        img_bird = render_rgb(self.env, camera=cfg.second_camera)
        bbox_result = self.planner.query_bbox(img, task, extra_images={"birdview": img_bird}, max_retries=3)
        if bbox_result is None:
            return None, None
        bbox, bbox_reasoning = bbox_result
        flipped = unflip_bbox(bbox)
        object_3d = bbox_to_3d(flipped, cfg.camera_size, self.env)
        if object_3d is None:
            return None, None
        # Show bbox detection frame (green box, 1 second pause)
        if self.on_frame:
            bbox_msg = f"Qwen bbox: {bbox}"
            if bbox_reasoning:
                bbox_msg += f"  |  {bbox_reasoning[:120]}"
            self.on_frame(render_rgb(self.env), "Bbox Detection", None, None,
                          bbox_msg, self._step, pause=True,
                          img2=render_rgb(self.env, camera=cfg.second_camera), bbox=bbox)
        return object_3d, bbox

    def run_closedloop(self, task: str) -> TrialResult:
        n = cfg.plan_length_n
        max_st = cfg.max_steps

        # Phase 1: Detect bbox → 3D (initial)
        object_3d, bbox = self._detect_and_project(task)
        if object_3d is None:
            return TrialResult(False, 0, "no_bbox_detected")
        self._bbox = bbox

        # Phase 2: Qwen plans actions with 3D context
        img = render_rgb(self.env)
        img_bird = render_rgb(self.env, camera=cfg.second_camera)
        plan = self.planner.plan_initial(img, task, object_3d, extra_images={"birdview": img_bird}, max_retries=4)
        if plan is None:
            return TrialResult(False, 0, "vlm_plan_failed")

        if self.on_frame:
            self.on_frame(render_rgb(self.env), "Initial VLM Plan",
                          plan, object_3d,
                          f"object at ({object_3d[0]:.3f}, {object_3d[1]:.3f})", self._step, pause=True,
                          img2=render_rgb(self.env, camera=cfg.second_camera))

        action_buffer = list(plan["actions"])
        action_count = 0
        completed_actions = []
        while self._step < max_st:
            cube_z = self.env.sim.data.body_xpos[self.env.cube_body_id][2]
            if cube_z > self._tz + 0.05:
                return TrialResult(True, self._step, None, plan)

            if len(action_buffer) == 0:
                # Detect bbox again for fresh 3D position
                object_3d, bbox = self._detect_and_project(task)
                if object_3d is None:
                    return TrialResult(False, self._step, "replan_bbox_failed")
                self._bbox = bbox

                ee = self._ee_pos()
                img = render_rgb(self.env)
                img_bird = render_rgb(self.env, camera=cfg.second_camera)
                grip_state = "closed (cube still on table — grasp may have failed)" \
                    if self._action_type in ("grasp", "lift") and self._action_count > 0 else "open"
                import time as _time

                _t0 = _time.time()
                new_plan = self.planner.plan_closedloop(
                    img, task, self._step, completed_actions, ee, n,
                    object_3d, gripper_state=grip_state,
                    extra_images={"birdview": img_bird})
                _elapsed = _time.time() - _t0
                if new_plan is not None and len(new_plan["actions"]) > 0:
                    print(f"    Got {len(new_plan['actions'])} actions in {_elapsed:.0f}s", flush=True)
                    action_buffer = list(new_plan["actions"])
                    action_count = 0
                    if self.on_frame:
                        self.on_frame(render_rgb(self.env), "Re-plan",
                                      new_plan, object_3d,
                                      f"{len(action_buffer)} actions ({_elapsed:.0f}s)", self._step, pause=True,
                                      img2=render_rgb(self.env, camera=cfg.second_camera))
                else:
                    print(f"  VLM re-plan failed ({_elapsed:.0f}s), retrying...", flush=True)
                    continue

            action = action_buffer.pop(0)
            action_count += 1
            atype = action["type"]
            params = {k: v for k, v in action.items() if k != "type"}
            self._action_count = action_count
            self._action_type = atype
            if self.on_frame:
                self.on_frame(render_rgb(self.env), f"Move #{action_count}: {atype}",
                              None, object_3d,
                              f"params={json.dumps(params)}", self._step, img2=render_rgb(self.env, camera=cfg.second_camera))
            result_str = self._execute_action(action, object_3d)
            completed_actions.append(f"{atype}(step={self._step},{result_str})")
        cube_z = self.env.sim.data.body_xpos[self.env.cube_body_id][2]
        return TrialResult(cube_z > self._tz + 0.05, self._step,
                           "timeout" if cube_z <= self._tz + 0.05 else None, plan)

print("VLM controllers defined")

VLM controllers defined


---
## 9. Diagnostic Check

In [13]:
def diagnostic_check():
    sep = "=" * 60
    print(sep)
    print("VLM DIAGNOSTIC CHECK")
    print(sep)

    print("\n--- 1. Configuration ---")
    print(f"  VLM model: {cfg.vlm_model_id}")
    print(f"  Camera:    {cfg.display_camera}")
    print(f"  Size:      {cfg.camera_size}")
    print(f"  m/n:       {cfg.feedback_interval_m}/{cfg.plan_length_n}")

    print("\n--- 2. Environment ---")
    env = make_env(cfg)
    env.reset()
    adjust_birdview_fov(env)
    cube_pos = env.sim.data.body_xpos[env.cube_body_id].copy()
    print(f"  Cube pos: ({cube_pos[0]:.3f}, {cube_pos[1]:.3f}, {cube_pos[2]:.3f})")

    print("\n--- 3. Camera Calibration ---")
    K, cam_pos, cam_R, tz = get_camera_params(env)
    print(f"  Position:  ({cam_pos[0]:.3f}, {cam_pos[1]:.3f}, {cam_pos[2]:.3f})")
    print(f"  Focal:     {K[0,0]:.1f}")
    print(f"  Table z:   {tz:.3f}")
    img_pt = K @ (cam_R.T @ (cube_pos - cam_pos))
    u, v = img_pt[0] / img_pt[2], img_pt[1] / img_pt[2]
    print(f"  Cube -> image: ({u:.0f}, {v:.0f})")

    print("\n--- 4. Phase 1: Qwen bbox detection ---")
    planner = get_planner()
    img = render_rgb(env)
    img_bird = render_rgb(env, camera=cfg.second_camera)
    task = "pick up the red cube"
    bbox_result = planner.query_bbox(img, task, extra_images={"birdview": img_bird}, max_retries=3)
    if bbox_result is None:
        print("  BBOX DETECTION FAILED")
        env.close()
        return
    bbox, bbox_reasoning = bbox_result
    print(f"  Bbox reasoning: {bbox_reasoning[:120] if bbox_reasoning else 'none'}")
    print(f"  Bbox: {bbox}")

    # Convert to 3D
    flipped = unflip_bbox(bbox)
    if flipped is None:
        print("  UNFLIP BBOX FAILED")
        env.close()
        return
    object_3d = bbox_to_3d(flipped, cfg.camera_size, env)
    if object_3d is None:
        print("  3D PROJECTION FAILED")
        env.close()
        return
    xy_err = np.linalg.norm(object_3d[:2] - cube_pos[:2])
    print(f"\n  Detected 3D: ({object_3d[0]:.3f}, {object_3d[1]:.3f}, {object_3d[2]:.3f})")
    print(f"  True cube:   ({cube_pos[0]:.3f}, {cube_pos[1]:.3f}, {cube_pos[2]:.3f})")
    print(f"  XY error:    {xy_err*100:.1f} cm")

    print("\n--- 5. Phase 2: Qwen action plan ---")
    plan = planner.plan_initial(img, task, object_3d, extra_images={"birdview": img_bird}, max_retries=4)
    if plan is None:
        print("  VLM PLAN FAILED")
    else:
        print(f"  Actions: {len(plan['actions'])} primitives")
        for i, a in enumerate(plan["actions"]):
            print(f"    {i+1}. {a}")

    eef = env.robots[0].eef_site_id
    ee_site = eef[list(eef.keys())[0]] if isinstance(eef, dict) else eef
    ee_pos = env.sim.data.site_xpos[ee_site].copy()
    print(f"\n  EE pos: ({ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f})")

    env.close()
    print(f"\n{sep}\nDiagnostic complete.\n{sep}")

if torch.cuda.is_available():
    diagnostic_check()
else:
    print("Skip diagnostic: no GPU")

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


VLM DIAGNOSTIC CHECK

--- 1. Configuration ---
  VLM model: Qwen/Qwen3-VL-4B-Instruct
  Camera:    agentview
  Size:      384
  m/n:       4/4

--- 2. Environment ---


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

  Birdview: Z=3.000, table Z=0.800, offset=1.2m
  Birdview FOV: 45.0° -> 21.3° (zoom=2.20x)
  Cube pos: (0.024, 0.017, 0.830)

--- 3. Camera Calibration ---
  Position:  (0.500, 0.000, 1.350)
  Focal:     463.5
  Table z:   0.800
  Cube -> image: (181, 213)

--- 4. Phase 1: Qwen bbox detection ---
Loading Qwen3-VL: Qwen/Qwen3-VL-4B-Instruct ...


config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

  VLM loaded
  [VLM] raw:
{"reasoning":"Red cube visible in both views, located centrally on table in birdview, and directly below robot's end-effector in agentview.","object_bbox":[0.49, 0.49, 0.51, 0.51]}

  [BBOX] Red cube visible in both views, located centrally on table in birdview, and directly below robot's end-effector in agent
  [BBOX] [0.49, 0.49, 0.51, 0.51]
  Bbox reasoning: Red cube visible in both views, located centrally on table in birdview, and directly below robot's end-effector in agent
  Bbox: [0.49, 0.49, 0.51, 0.51]

  Detected 3D: (-0.042, 0.000, 0.800)
  True cube:   (0.024, 0.017, 0.830)
  XY error:    6.8 cm

--- 5. Phase 2: Qwen action plan ---
  [VLM] raw:
{"reasoning":"The red cube is on the table, and the robot arm is positioned above it. I need to descend to press down on the cube, then grasp it. The birdview shows the cube is centered in the workspace, so I will hover above it first, then descend and grasp.","actions":[{"type":"move_linear","target":"obj

---
## 10. Video Demo

---
## Experiment Configuration

Configure experiment parameters before running the video demo.

- **`task_type`**: Category for logging (`"pick_up"`, `"push"`, `"reach"`, `"custom"`, etc.)
- **`user_prompt`**: Natural language instruction sent to the VLM. May be completely unrelated to the environment -- the VLM must handle arbitrary input.
- **`use_feedback`**: `True` = closed-loop, `False`removed (always closed-loop)
- **`video_path`**: Output path for the annotated video


In [14]:
# ============================================================
# Experiment Config -- edit before running Cell 10
# ============================================================
task_type = "pick_up"              # Category for logging
user_prompt = "pick up the red cube"  # NL instruction for VLM
# use_feedback removed (always closed-loop)
video_path = "output.mp4"          # Output video path
# ============================================================

print(f"Config: task_type={task_type}")
print(f"  user_prompt='{user_prompt}'")
print(f"  video_path={video_path}")



Config: task_type=pick_up
  user_prompt='pick up the red cube'
  video_path=output.mp4


In [15]:
def record_video(task, video_path):
    cam = VideoCapture(fps=4)
    planner = get_planner()
    env = make_env(cfg)
    env.reset()
    adjust_birdview_fov(env)

    cam.snap(render_rgb(env), "Initial", None, None, f'Task: "{task}"',
             img2=render_rgb(env, camera=cfg.second_camera))

    ctrl = VLMController(env, planner, on_frame=cam.snap)
    from tqdm.notebook import tqdm
    pbar = tqdm(total=cfg.max_steps, desc="Steps", unit="step")
    _orig_snap = cam.snap
    def _snap_with_pbar(img, phase="", plan=None, t3d=None, msg="", step=0, pause=False, img2=None, bbox=None):
        try:
            pbar.n = min(step, cfg.max_steps)
            pbar.refresh()
            _orig_snap(img, phase, plan, t3d, msg, step, pause, img2=img2, bbox=bbox)
        except Exception as e:
            print(f"  [FRAME] snap error: {e}", flush=True)
    ctrl.on_frame = _snap_with_pbar
    result = ctrl.run_closedloop(task)
    pbar.n = min(ctrl._step, cfg.max_steps)
    pbar.refresh()
    pbar.close()

    final_msg = "SUCCESS" if result.success else f"FAILED ({result.error})"
    cam.snap(render_rgb(env), final_msg, result.plan, None, f"Steps: {result.steps}", step=result.steps,
             img2=render_rgb(env, camera=cfg.second_camera))

    env.close()
    print(f"  [RECORD] Frames captured: {len(cam.frames)}", flush=True)
    path = cam.save(video_path)
    return path, result


if torch.cuda.is_available():
    path, result = record_video(
        task=user_prompt,
        video_path=video_path
    )
    print(f"Video saved: {path}")
    print(f"Result: {'SUCCESS' if result.success else 'FAIL'} ({result.error or 'ok'})")
    from IPython.display import Video
    try:
        display(Video(path, width=960, height=680))
    except Exception:
        print("Video display unavailable - file saved for download.")
else:
    print("Skip video: no GPU")

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but 

  Birdview: Z=3.000, table Z=0.800, offset=1.2m
  Birdview FOV: 45.0° -> 21.3° (zoom=2.20x)


Steps:   0%|          | 0/150 [00:00<?, ?step/s]

  [VLM] raw:
{"reasoning":"Red cube visible in both views, located centrally on table in birdview, and directly below robot arm in agentview.","object_bbox":[0.49, 0.49, 0.51, 0.51]}

  [BBOX] Red cube visible in both views, located centrally on table in birdview, and directly below robot arm in agentview.
  [BBOX] [0.49, 0.49, 0.51, 0.51]
  [VLM] raw:
{"reasoning":"The red cube is on the table and the robot arm is positioned above it. I need to move the end-effector down to hover above the cube, then descend to press down, grasp it, and lift it off the table.","actions":[{"type":"move_linear","target":"object_above","offset":[0,0,0.1],"rotation":[0,0,0],"speed":0.3,"max_steps":30},{"type":"descend","speed":0.3,"max_steps":50},{"type":"grasp","force":0.8,"max_steps":30},{"type":"lift","height":0.2,"speed":0.5,"max_steps":40}]}

  [PLAN] reasoning: The red cube is on the table and the robot arm is positioned above it. I need to move the end-effector down to hover above the cube, then de

[ERROR:0@348.588] global cap_ffmpeg_impl.hpp:3203 open Could not find encoder for codec_id=27, error: Encoder not found
[ERROR:0@348.588] global cap_ffmpeg_impl.hpp:3281 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter
OpenCV: FFMPEG: tag 0x34363248/'H264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'
[ERROR:0@348.590] global cap_ffmpeg_impl.hpp:3203 open Could not find encoder for codec_id=27, error: Encoder not found
[ERROR:0@348.590] global cap_ffmpeg_impl.hpp:3281 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter
OpenCV: FFMPEG: tag 0x34363258/'X264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'
[ERROR:0@348.590] global cap_ffmpeg_impl.hpp:3203 open Could not find encoder for codec_id=27, error: Encoder not found
[ERROR:0@348.590] global cap_ffmpeg_impl.hpp:3281 open VIDEOIO/FFMPEG: Failed to initialize VideoWri

  [SAVE] Written with mp4v: 1397 KB
Video saved: output.mp4
Result: FAIL (timeout)


---
## 11. Batch Evaluation

In [16]:
def evaluate_vlm(n: int, planner, task="pick up the red cube") -> list:
    label = "Closed-Loop VLM"
    sep = "=" * 40
    print(f"\n{sep}\n{label} Evaluation ({n} trials)\n{sep}")
    results = []
    for trial in range(n):
        print(f"  Trial {trial+1}/{n} ...", end=" ")
        env = make_env(cfg, seed=cfg.seed + trial)
        env.reset()
        adjust_birdview_fov(env)
        ctrl = VLMController(env, planner)
        result = ctrl.run_closedloop(task)
        results.append(result)
        env.close()
        print(f"{'OK' if result.success else 'FAIL'} ({result.error or 'ok'}, {result.steps} steps)")
    return results


def summary(results, label):
    n, ok = len(results), sum(1 for r in results if r.success)
    errs = {}
    for r in results:
        k = r.error or "ok"
        errs[k] = errs.get(k, 0) + 1
    print(f"\n{'='*40}\n{label}: {ok}/{n} = {100*ok/n:.1f}%")
    for e, c in sorted(errs.items()):
        print(f"  {e}: {c}")
    print("=" * 40)
    return ok / n


if torch.cuda.is_available():
    # n = min(cfg.n_trials, 5)
    n = 100
    print("Loading VLM planner (shared across all trials)...")
    planner = get_planner()

    # Enable incremental VLM log
    import os as _os
    _log_path = "vlm_batch_log.txt"
    vlm_log_path = _log_path
    if _os.path.exists(_log_path):
        _os.remove(_log_path)

    results = evaluate_vlm(n, planner)
    vlm_log_path = None
    print(f"VLM log written to {_log_path}")

    rc = summary(results, "Closed-Loop VLM")

    torch.save({"closed_loop": results}, "vlm_eval_results.pt")
    print("Saved to vlm_eval_results.pt")
else:
    print("Skip: no GPU")

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


Loading VLM planner (shared across all trials)...

Closed-Loop VLM Evaluation (100 trials)
  Trial 1/100 ... 

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

  Birdview: Z=3.000, table Z=0.800, offset=1.2m
  Birdview FOV: 45.0° -> 21.3° (zoom=2.20x)
  [VLM] raw:
{"reasoning":"Red cube visible in both views, located centrally on table in birdview, and directly below robot arm in agentview.","object_bbox":[0.48,0.48,0.52,0.52]}

  [BBOX] Red cube visible in both views, located centrally on table in birdview, and directly below robot arm in agentview.
  [BBOX] [0.48, 0.48, 0.52, 0.52]
  [VLM] raw:
{"reasoning":"The red cube is on the table and the robot arm is positioned above it. I need to descend to press down on the cube, then grasp it to pick it up.","actions":[{"type":"move_linear","target":"object_above","offset":[0,0,0.1],"rotation":[0,0,0],"speed":0.3,"max_steps":30},{"type":"descend","speed":0.3,"max_steps":50},{"type":"grasp","force":0.8,"max_steps":40}]}

  [PLAN] reasoning: The red cube is on the table and the robot arm is positioned above it. I need to descend to press down on the cube, then grasp it to pick it up.
  [PLAN] 3 acti

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


...

  Birdview: Z=3.000, table Z=0.800, offset=1.2m
  Birdview FOV: 45.0° -> 21.3° (zoom=2.20x)
  [VLM] raw:
{"reasoning":"Red cube visible in both views, centered on table. Agent view confirms position from above. Cross-check with bird view shows object at center. Bounding box in agent view: [0.45,0.45,0.55,0.55].","object_bbox":[0.45,0.45,0.55,0.55]}

  [BBOX] Red cube visible in both views, centered on table. Agent view confirms position from above. Cross-check with bird view s
  [BBOX] [0.45, 0.45, 0.55, 0.55]
  [VLM] raw:
{"reasoning":"The red cube is on the table and the robot arm is positioned above it. I need to descend to press down on the cube, then grasp it, and lift it off the table.","actions":[{"type":"move_linear","target":"object_above","offset":[0,0,0.1],"rotation":[0,0,0],"speed":0.3,"max_steps":30},{"type":"descend","speed":0.3,"max_steps":50},{"type":"grasp","force":0.8,"max_steps":40}]}

  [PLAN] reasoning: The red cube is on the table and the robot arm is positioned a

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)


OK (ok, 146 steps)
  Trial 100/100 ... 

[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

  Birdview: Z=3.000, table Z=0.800, offset=1.2m
  Birdview FOV: 45.0° -> 21.3° (zoom=2.20x)
  [VLM] raw:
{"reasoning":"Red cube visible in both views, located centrally on table in birdview, and directly under robot arm in agentview.","object_bbox":[0.49, 0.49, 0.51, 0.51]}

  [BBOX] Red cube visible in both views, located centrally on table in birdview, and directly under robot arm in agentview.
  [BBOX] [0.49, 0.49, 0.51, 0.51]
  [VLM] raw:
{"reasoning":"The red cube is on the table at the target position. The robot arm is positioned above it in the agent view, and the bird view confirms its location. The next step is to descend to press down on the cube, then grasp it.","actions":[{"type":"move_linear","target":"object_above","offset":[0,0,0.1],"rotation":[0,0,0],"speed":0.3,"max_steps":30},{"type":"descend","speed":0.3,"max_steps":50},{"type":"grasp","force":0.8,"max_steps":40}]}

  [PLAN] reasoning: The red cube is on the table at the target position. The robot arm is positioned a

---
## Model Cleanup


In [17]:
# Run this cell to unload the VLM and free GPU memory
if '_vlm_planner' in globals() and globals()['_vlm_planner'] is not None:
    _vlm_planner.unload()
    _vlm_planner = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print('VLM unloaded, GPU memory freed')
else:
    print('No VLM loaded')



  VLM unloaded
VLM unloaded, GPU memory freed


---
## 12. Failure Mode Analysis

In [18]:
def analyze_vlm(path="vlm_eval_results.pt"):
    if not os.path.exists(path):
        print("No saved results. Run Cell 11 first.")
        return
    data = torch.load(path, weights_only=False)
    for name, key in [("Closed-Loop VLM", "closed_loop")]:
        results = data[key]
        summary(results, name)
        modes = {}
        for r in results:
            if r.error:
                modes[r.error] = modes.get(r.error, 0) + 1
        if modes:
            print("  Failure breakdown:")
            for m, c in sorted(modes.items()):
                print(f"    {m}: {c}/{len(results)} ({100*c//len(results)}%)")

if __name__ == "__main__" or True:
    analyze_vlm()




Closed-Loop VLM: 25/100 = 25.0%
  no_bbox_detected: 2
  ok: 25
  replan_bbox_failed: 2
  timeout: 71
  Failure breakdown:
    no_bbox_detected: 2/100 (2%)
    replan_bbox_failed: 2/100 (2%)
    timeout: 71/100 (71%)


---
## Usage Notes

| Step | Action |
|------|--------|
| 1 | Enable **GPU T4 x2** in Kaggle Accelerator |
| 2 | Run cells **1 -> 12** in order |
| 3 | First run downloads Qwen3-VL-4B (~8 GB) — approx 2-3 min |
| 4 | **Cell 10** generates annotated video; **Cell 11** runs batch evaluation |
| 5 | Increase cfg.n_trials to 50+ in Cell 5 for publishable results |

### Parameter Tuning

| Parameter | Effect |
|-----------|--------|
| feedback_interval_m | Lower = more frequent VLM calls (slower but more reactive) |
| plan_length_n | Higher = longer lookahead (must be >= m) |
| lift_height | How high to lift the object above table |

### Common Issues

| Symptom | Fix |
|---------|-----|
| VLM plan fails/returns None | Rerun Cell 10; check VLM output |
| VLM bbox inaccurate | Qwen3-VL may need a more specific object description |
| Controller error | Robosuite 1.5 uses load_composite_controller_config("BASIC") |
| OOM | Set cfg.use_4bit = True (default) |
| Results lost | Download vlm_eval_results*.pt + output.mp4 from Kaggle working dir

### Architecture

```ascii
RGB Image + Task Description
         |
    [Qwen3-VL-4B]  ← outputs bbox + DSL actions in one JSON
         |
    bbox → Ray-Plane Intersection → 3D Position
         |
    DSL Action Plan (with "object_above" resolved to 3D)
         |
    OSC_POSE Controller
         |
    [Robot Execution]
         |
    (action buffer empty)
         |
    [Loop: Re-plan with updated 3D context]
```